# Gaia2 on a budget: reproduction, failure analysis, and a waiting-instruction extension
CS329Z Paper Video — Yating Zhu

Paper: Froger et al., *Gaia2: Benchmarking LLM Agents on Dynamic and Asynchronous Environments*, ICLR 2026. arXiv:2602.11964. Code: `meta-agents-research-environments` (ARE) 1.2.0.

What this notebook does
1. Runs the released Gaia2 harness on the Gaia2-mini validation split: 10 Execution + 10 Time scenarios, Gemini 2.5 Flash and Claude Sonnet 4.5, judge Llama-3.3-70B, 1 run each (Part 1).
2. Analyses the results: timeline plots of world events vs agent actions, re-judging the same traces with GPT-5-mini, a summary table (Part 2).
3. Extension: adds one waiting instruction to the agent's system prompt and re-runs the 10 Time scenarios; plus single-scenario live runs used in the video (Parts 2–3).

Runs on Google Colab. Needs two Colab Secrets: `HF_TOKEN` (Hugging Face, read) and `OPENROUTER_API_KEY`. All outputs go to `/content/results/`.

## Step 1: Install (3–5 min)
If Colab asks to restart the runtime, do so and continue from Step 2.

In [ ]:
%%capture --no-stderr
!pip install -q "meta-agents-research-environments==1.2.0" huggingface_hub pandas requests


In [ ]:
# Sanity check: should print the first lines of the help text
!are-benchmark --help | head -n 8


## Step 2: Load API keys from Colab Secrets
No keys are stored in this notebook.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
print("Keys loaded.")

## Step 2.5: Download the simulated phone's file system (259 MB, 2–3 min)
By default ARE lazily fetches each file from Hugging Face, which hit HF's rate limit (HTTP 429) in my first attempt. Downloading the snapshot once and pointing `DEMO_FS_PATH` at it avoids that.

In [ ]:
import os
from huggingface_hub import snapshot_download

local_fs = snapshot_download(
    repo_id="meta-agents-research-environments/gaia2_filesystem",
    repo_type="dataset",
    allow_patterns=["demo_filesystem/*", "demo_filesystem/**"],
    local_dir="/content/gaia2_fs",
)
os.environ["DEMO_FS_PATH"] = "/content/gaia2_fs/demo_filesystem"
print("File system downloaded to:", os.environ["DEMO_FS_PATH"])
print("Contents:", os.listdir(os.environ["DEMO_FS_PATH"]))


## Step 2.6: Restore earlier results (only in a new Colab session)
Upload a previous `gaia2_results*.zip` to `/content` first; the newest one is restored.

In [ ]:
import zipfile, os, glob
from pathlib import Path
zips = sorted(glob.glob("/content/gaia2_results*.zip"), key=os.path.getmtime)
if zips:
    Path("/content/results").mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[-1]) as z: z.extractall("/content/results")
    print("Restored from", zips[-1], ":", sorted(os.listdir("/content/results")))
else:
    print("No /content/gaia2_results*.zip found. Skip this step if this is the first session.")

## Step 3: Check OpenRouter balance (before and after, to measure cost)

In [ ]:
import requests, os

r = requests.get("https://openrouter.ai/api/v1/key",
                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}, timeout=30)
r.raise_for_status()
d = r.json().get("data", r.json())
limit = d.get("limit"); usage = d.get("usage")
print(f"limit = {limit}  usage = {usage}")
if limit is not None and usage is not None:
    print(f"remaining ≈ ${limit - usage:.2f}")


## Step 4: Run helper
`run_gaia2(config, model, limit, run_name)` runs `are-benchmark run` on the Gaia2 validation split.
- `config`: `execution`, `search`, `ambiguity`, `adaptability`, `time`, or `mini`
- `model`: an OpenRouter model, e.g. `openrouter/google/gemini-2.5-flash`
- `limit`: number of scenarios; `run_name`: output folder under `/content/results/`

Judge: Llama-3.3-70B as in the paper. One run per scenario (the paper averages 3) to keep cost low. Scenarios run one at a time to avoid HF rate limits, with automatic retry.

In [ ]:
import subprocess, shlex, os, time
from pathlib import Path

RESULTS_ROOT = Path("/content/results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

JUDGE_MODEL = "openrouter/meta-llama/llama-3.3-70b-instruct"

def _run_once(cmd, out_dir):
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    rate_limited = False
    for line in proc.stdout:
        # HF rate-limit warnings are very noisy: print once, swallow the rest
        if "429 Client Error" in line or "Too Many Requests" in line or "hit the quota" in line:
            if not rate_limited:
                print("[note] Hit the Hugging Face rate limit (429); suppressing repeats.")
            rate_limited = True
            continue
        print(line, end="")
    proc.wait()
    print(f"\nexit code {proc.returncode}, {(time.time()-t0)/60:.1f} min")
    stats = (out_dir / "benchmark_stats.json").exists()
    return stats, rate_limited

def run_gaia2(config, model, limit, run_name, timeout=900, concurrency=1, num_runs=1, max_tries=3):
    out_dir = RESULTS_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "are-benchmark", "run",
        "--hf-dataset", "meta-agents-research-environments/gaia2",
        "--hf-split", "validation",
        "--hf-config", config,
        "--model", model, "--provider", "local",
        "--judge_model", JUDGE_MODEL, "--judge_provider", "local",
        "--agent", "default",
        "--num_runs", str(num_runs),
        "--limit", str(limit),
        "--max_concurrent_scenarios", str(concurrency),
        "--scenario_timeout", str(timeout),
        "--trace_dump_format", "both",
        "--output_dir", str(out_dir),
    ]
    print("Command:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    with open(out_dir / "command.sh", "w") as f:
        f.write(" ".join(shlex.quote(c) for c in cmd) + "\n")
    for attempt in range(1, max_tries + 1):
        print(f"===== attempt {attempt} =====")
        ok, rate_limited = _run_once(cmd, out_dir)
        if ok:
            print(f"Done. Results in {out_dir}")
            return out_dir
        if attempt < max_tries:
            print("No results (most likely the HF rate limit). Retrying in 5 minutes...")
            time.sleep(300)
    print("Failed three times; see the log above.")
    return out_dir

print("Helper defined.")


## Step 5: Smoke test — 1 Execution scenario
Success = a Validation Report and exit code 0. Warnings like `Setting max_turns to 1` are normal.

In [ ]:
run_gaia2(config="execution", model="openrouter/google/gemini-2.5-flash", limit=1, run_name="test_1")


## Step 6: Result loaders
Reads the per-scenario `lite` JSON (decision is `Valid` / `Invalid`) and the official `benchmark_stats.json`.

In [ ]:
import json, glob
import pandas as pd
from pathlib import Path

def load_run(run_name):
    # One row per scenario from the lite JSON. Decision is Valid / Invalid
    rows = []
    for fp in glob.glob(str(RESULTS_ROOT / run_name / "**" / "lite" / "*.json"), recursive=True):
        try:
            d = json.load(open(fp))
        except Exception:
            continue
        if "scenario_id" not in d:
            continue
        stats = d.get("per_agent_llm_usage_stats", {}) or {}
        main = next(iter(stats.values()), {}) if stats else {}
        rows.append({
            "run": run_name,
            "scenario_id": d.get("scenario_id"),
            "model": d.get("model_id"),
            "decision": d.get("validation_decision"),
            "passed": str(d.get("validation_decision")).lower() == "valid",
            "llm_calls": main.get("total_llm_calls"),
            "rationale": (d.get("validation_rationale") or "")[:120],
            "file": fp,
        })
    return pd.DataFrame(rows)

def run_stats(run_name):
    # Official stats file: success rate and average duration
    fp = RESULTS_ROOT / run_name / "benchmark_stats.json"
    if not fp.exists():
        return None
    s = json.load(open(fp))
    g = s.get("global", {}) or {}
    out = {"run": run_name,
           "success_rate_%": g.get("macro_success_rate"),
           "avg_run_duration_s": g.get("avg_run_duration")}
    # Fallback: look for a per-capability block if 'global' is missing
    def walk(o):
        if isinstance(o, dict):
            if "capability" in o:
                yield o
            for v in o.values():
                yield from walk(v)
        elif isinstance(o, list):
            for v in o:
                yield from walk(v)
    caps = list(walk(s))
    if caps and out["success_rate_%"] is None:
        out["success_rate_%"] = caps[0].get("success_rate")
        out["avg_run_duration_s"] = caps[0].get("avg_run_duration")
    return out

def summarize(run_names):
    table = []
    frames = []
    for r in run_names:
        df = load_run(r)
        frames.append(df)
        st = run_stats(r) or {"run": r}
        st["n"] = len(df)
        st["passed"] = int(df["passed"].sum()) if len(df) else 0
        st["pass_rate_%_from_files"] = round(100 * df["passed"].mean(), 1) if len(df) else None
        st["avg_llm_calls"] = round(df["llm_calls"].mean(), 1) if len(df) else None
        table.append(st)
    summary = pd.DataFrame(table).set_index("run")
    display(summary)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df_test = summarize(["test_1"])
df_test[["scenario_id", "decision", "llm_calls", "rationale"]]


# Part 1: Reproduction
## Step 7: 10 Execution scenarios, Gemini 2.5 Flash

In [ ]:
run_gaia2(config="execution", model="openrouter/google/gemini-2.5-flash", limit=10, run_name="exec_flash")
df = summarize(["exec_flash"])


## Step 8: 10 Time scenarios, Gemini 2.5 Flash
The paper reports Time as the weakest capability for every model.

In [ ]:
run_gaia2(config="time", model="openrouter/google/gemini-2.5-flash", limit=10, run_name="time_flash")
df = summarize(["exec_flash", "time_flash"])


## Step 9: Same 20 scenarios with a slower, stronger model (Claude Sonnet 4.5)
Tests the paper's claim that stronger-but-slower models do worse on Time.

In [ ]:
STRONG = "openrouter/anthropic/claude-sonnet-4.5"

run_gaia2(config="execution", model=STRONG, limit=10, run_name="exec_strong")
run_gaia2(config="time",      model=STRONG, limit=10, run_name="time_strong")
df = summarize(["exec_flash", "time_flash", "exec_strong", "time_strong"])


## Step 10: Summary of all runs

In [ ]:
all_runs = [p.name for p in RESULTS_ROOT.iterdir() if p.is_dir()]
print("Runs found:", all_runs)
df_all = summarize(all_runs)


## Step 11: Print one trace step by step
Change `RUN` and `INDEX` to inspect other failures.

In [ ]:
RUN = "exec_strong"  # which run
INDEX = 0            # which failed scenario

df_run = load_run(RUN)
failed = df_run[~df_run["passed"]].reset_index(drop=True)
print(f"{RUN}: {len(df_run)} scenarios, {len(failed)} failed\n")
if len(failed) == 0:
    print("No failures in this run.")
else:
    row = failed.iloc[INDEX]
    d = json.load(open(row["file"]))
    print("scenario_id:", d.get("scenario_id"))
    print("decision:", d.get("validation_decision"))
    print("rationale:", d.get("validation_rationale"))
    print("=" * 80)
    hist = d.get("per_agent_interaction_histories", {}) or {}
    main_hist = next(iter(hist.values()), []) if hist else []
    for i, m in enumerate(main_hist):
        content = str(m.get("content", ""))
        if len(content) > 600:
            content = content[:600] + " …[truncated]"
        print(f"[{i}] {m.get('role')}:\n{content}\n")


## Step 12: Check balance again (difference = cost)

In [ ]:
r = requests.get("https://openrouter.ai/api/v1/key",
                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}, timeout=30)
d = r.json().get("data", r.json())
print(f"limit {d.get('limit')}  usage {d.get('usage')}  remaining ≈ ${(d.get('limit') or 0) - (d.get('usage') or 0):.2f}")


## Step 13: Download all results

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/gaia2_results", "zip", RESULTS_ROOT)
print("Packed:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Not in Colab; download the file from the path above.")


# Part 2: Analysis and extension
These cells read the saved results and take seconds, except Step 15 (a few minutes) and Step 17 (about an hour).

## Step 14: Timeline plots — what happened in the world vs what the agent did
Red ▼ = environment events (a new listing, a reply). Grey bar = agent waiting. Green = agent write actions. Hollow green ▲ = when the oracle expected the agent to act. Saved to `/content/results/figs/`.

In [ ]:
import json, glob
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path("/content/results/figs"); FIG_DIR.mkdir(parents=True, exist_ok=True)

SHORT = {
    "send_message": "send message", "send_message_to_group_conversation": "send group msg",
    "send_message_to_user": "report to user", "wait_for_notification": "wait",
    "search_apartments": "search", "save_apartment": "save apt", "remove_saved_apartment": "unsave apt",
    "add_new_apartment": "new listing", "create_and_add_message": "reply arrives",
    "add_calendar_event": "add calendar", "send_email": "send email",
}
def _fn(a): return (a or {}).get("function") or ""
def _name(a): return SHORT.get(_fn(a), _fn(a) or "?")
def _arg(a, key):
    for x in (a or {}).get("args") or []:
        if isinstance(x, dict) and x.get("name") == key: return str(x.get("value"))
    return ""

def load_trace(path):
    d = json.load(open(path))
    done = [e for e in d["completed_events"] if e.get("event_time") is not None]
    t0 = min(e["event_time"] for e in done)
    agent = [(e["event_time"] - t0, e["action"]) for e in done if e["event_type"] == "AGENT"]
    env = [(e["event_time"] - t0, e["action"]) for e in done if e["event_type"] == "ENV"]
    happened = {e["event_id"] for e in done}
    env_time = {e["event_id"]: e["event_time"] - t0 for e in done if e["event_type"] == "ENV"}
    missed_env = []
    for e in d["events"]:
        if e["event_type"] == "ENV" and e["event_id"] not in happened:
            missed_env.append((e.get("event_relative_time") or 0, e["action"]))
            env_time.setdefault(e["event_id"], e.get("event_relative_time") or 0)
    expected = []
    for e in d["events"]:
        if e["class_name"] != "OracleEvent": continue
        deps = [x for x in (e.get("dependencies") or []) if isinstance(x, str) and x.startswith("Event-ENV")]
        if deps and _fn(e["action"]) in ("send_message","send_email","remove_saved_apartment","add_calendar_event","send_message_to_group_conversation"):
            expected.append((env_time.get(deps[0], 0) + (e.get("event_relative_time") or 0), e["action"]))
    meta = d["metadata"]
    ann = meta.get("annotation") if isinstance(meta.get("annotation"), dict) else {}
    return dict(agent=agent, env=env, missed_env=missed_env, expected=expected,
                decision=ann.get("validation_decision"), model=(meta.get("simulation") or {}).get("model_id",""),
                sid=(meta.get("definition") or {}).get("scenario_id", Path(path).stem))

def plot_timeline(path, title=None, xmax=None, save_as=None):
    r = load_trace(path)
    fig, ax = plt.subplots(figsize=(13, 5))
    for i, (t, a) in enumerate(sorted(r["env"], key=lambda x: x[0])):
        ax.plot([t], [1], marker="v", ms=11, color="#d62728", ls="none")
        price = _arg(a, "price")
        label = (_arg(a, "name") or _arg(a, "content")[:18]) + (f"  ${float(price):.0f}" if price not in ("", "None") else "")
        ax.annotate(f"{_name(a)}\n{label}", (t, 1), xytext=(0, 12 + 22 * (i % 3)), textcoords="offset points", ha="center", fontsize=8, color="#d62728")
    for t, a in r["missed_env"]:
        ax.plot([t], [1], marker="v", ms=11, color="#d62728", ls="none", alpha=0.35)
        ax.annotate(f"scripted, never reached:\n{_name(a)} at +{t:.0f}s", (t, 1), xytext=(0, 12), textcoords="offset points", ha="center", fontsize=8, color="#d62728", alpha=0.8)
    for t, a in r["expected"]:
        ax.plot([t], [0], marker="^", ms=10, mfc="none", mec="#2ca02c", mew=1.6, ls="none")
        ax.annotate(f"expected: {_name(a)}", (t, 0), xytext=(0, 14), textcoords="offset points", ha="center", fontsize=8, color="#2ca02c")
    last_t = 0
    for t, a in r["agent"]:
        fn = _fn(a); color, marker = "#1f77b4", "o"
        if fn == "wait_for_notification":
            color, marker = "#7f7f7f", "s"
            ax.plot([t, t + float(_arg(a, "timeout") or 0)], [0, 0], color="#7f7f7f", lw=6, alpha=0.35, solid_capstyle="butt")
        if fn in ("send_message","send_message_to_group_conversation","send_email","remove_saved_apartment","add_calendar_event"): color = "#2ca02c"
        if fn == "send_message_to_user": color = "#ff7f0e"
        ax.plot([t], [0], marker=marker, ms=8, color=color, ls="none")
        if fn in ("send_message","send_message_to_group_conversation","send_email","remove_saved_apartment","add_calendar_event","send_message_to_user"):
            ax.annotate(_name(a), (t, 0), xytext=(0, -22), textcoords="offset points", ha="center", fontsize=8, color=color)
        last_t = max(last_t, t)
    ax.axvline(last_t, color="#ff7f0e", ls="--", lw=1)
    ax.text(last_t, 0.5, f"  agent stops at t+{last_t:.0f}s", color="#ff7f0e", fontsize=9, va="center")
    ax.set_yticks([0, 1]); ax.set_yticklabels(["what the agent did", "what happened in the world"])
    ax.set_xlabel("simulated time (seconds)")
    allt = [t for t, _ in r["agent"]] + [t for t, _ in r["env"]] + [t for t, _ in r["missed_env"]] + [t for t, _ in r["expected"]]
    ax.set_xlim(-5, xmax or (max(allt) + 30)); ax.set_ylim(-0.8, 2.3); ax.grid(axis="x", alpha=0.3)
    ax.set_title(title or f"{r['sid']} | {r['model'].split('/')[-1]} | verdict: {r['decision']}", fontsize=11)
    fig.tight_layout()
    out = FIG_DIR / (save_as or f"{r['sid']}_{r['model'].split('/')[-1]}.png")
    fig.savefig(out, dpi=160); plt.show()
    print("saved", out)
    return r

def trace_path(run, sid):
    hits = glob.glob(f"/content/results/{run}/hf/{sid}*.json")
    assert hits, f"not found: {run}/{sid}"
    return hits[0]

print("Plot helpers defined.")


### 14a. Three cases: waited but late (two models), and did not wait at all

In [ ]:
# Fig 1: xvc7uo, "notify Linnea whenever an in-budget listing appears". Claude waited (60 s polls), messaged the right listings, still Invalid
plot_timeline(trace_path("time_strong", "scenario_universe_21_xvc7uo"), xmax=270,
              title='"Notify Linnea immediately whenever an in-budget listing appears" | Claude Sonnet 4.5 | verdict: Invalid')
# Fig 2: same scenario, Gemini Flash (30 s polls)
plot_timeline(trace_path("time_flash", "scenario_universe_21_xvc7uo"), xmax=270,
              title='Same task | Gemini 2.5 Flash | verdict: Invalid')
# Fig 3: wezjyh, "ask Yujie, wait up to 4 min for her reply". The agent ended its turn without waiting
plot_timeline(trace_path("time_flash", "scenario_universe_21_wezjyh"), xmax=200,
              title='"Ask Yujie, wait up to 4 min for her reply, then act" | Gemini 2.5 Flash | verdict: Invalid')


## Step 15: Re-judge the same traces with a different judge
No agent is re-run; only the verdicts are recomputed. Original judge: Llama-3.3-70B. New judge: GPT-5-mini. Output: a per-scenario comparison and the number of flipped verdicts.

In [ ]:
import subprocess, shlex, time, json, glob
from pathlib import Path
import pandas as pd

NEW_JUDGE = "openrouter/openai/gpt-5-mini"

def rejudge(run_name, new_judge=NEW_JUDGE, out_suffix="rejudge", concurrency=5, timeout=600):
    src_dir = Path(f"/content/results/{run_name}/hf")
    out_dir = Path(f"/content/results/{run_name}_{out_suffix}"); out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["are-benchmark", "judge", "-d", str(src_dir),
           "--model", "openrouter/google/gemini-2.5-flash", "--provider", "local",   # required arg; not called when only judging
           "--judge_model", new_judge, "--judge_provider", "local",
           "--max_concurrent_scenarios", str(concurrency), "--scenario_timeout", str(timeout),
           "--trace_dump_format", "both", "--output_dir", str(out_dir)]
    print("Command:\n  " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    (out_dir / "command.sh").write_text(" ".join(shlex.quote(c) for c in cmd) + "\n")
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "429" in line or "lazy load" in line: continue
        if any(k in line for k in ("Result:", "Validation Report", "success rate", "ERROR", "Traceback", "Error")):
            print(line, end="")
    p.wait(); print(f"\nexit code {p.returncode}, {(time.time()-t0)/60:.1f} min. Results in {out_dir}")
    return out_dir

def decisions(run_name):
    out = {}
    for fp in glob.glob(f"/content/results/{run_name}/**/lite/*.json", recursive=True):
        d = json.load(open(fp)); out[d["scenario_id"]] = d.get("validation_decision")
    return out

def compare_judges(run_name, suffix="rejudge"):
    a, b = decisions(run_name), decisions(f"{run_name}_{suffix}")
    rows = [{"scenario": s, "Llama-3.3-70B judge": a.get(s), "new judge": b.get(s),
             "flipped": (a.get(s) != b.get(s))} for s in sorted(a)]
    df = pd.DataFrame(rows)
    n_valid_old = sum(v == "Valid" for v in a.values()); n_valid_new = sum(v == "Valid" for v in b.values())
    print(f"{run_name}: Llama judge {n_valid_old}/{len(a)} valid  ->  new judge {n_valid_new}/{len(b)} valid; {int(df['flipped'].sum())} verdicts flipped")
    display(df)
    return df

print("Re-judge helpers defined.")


In [ ]:
# Re-judge two runs: Time (Flash) and Execution (Claude), 10 scenarios each
rejudge("time_flash")
rejudge("exec_strong")
df_rj1 = compare_judges("time_flash")
df_rj2 = compare_judges("exec_strong")


## Step 16: Summary table (saved to `results/figs/summary.csv`)

In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for p in sorted(Path("/content/results").iterdir()):
    st = p / "benchmark_stats.json"
    if not st.exists(): continue
    s = json.load(open(st)); g = s["statistics"]["global"]; m = s["metadata"]
    rows.append({"run": p.name, "model": m.get("model", "").split("/")[-1], "n": g["total_runs"],
                 "passed": g["success_runs"], "success_rate_%": round(g["macro_success_rate"], 1),
                 "avg_duration_s": round(g["avg_run_duration"], 1)})
summary = pd.DataFrame(rows).set_index("run")
summary.to_csv("/content/results/figs/summary.csv")
display(summary)


## Step 17: Extension — can one instruction teach the agent to wait?
Adds a short paragraph to the default agent's system prompt (via `json_agent_hints`), re-runs the same 10 Time scenarios with Gemini 2.5 Flash, then restores the prompt. Compare `time_flash` vs `time_flash_wait`.

In [ ]:
import importlib, shutil, re
from pathlib import Path
import are.simulation.agents.default_agent.prompts.system_prompt as sp

PROMPT_FILE = Path(sp.__file__)
BACKUP = PROMPT_FILE.with_suffix(".py.bak")

WAITING_HINT = """
TIME-SENSITIVE TASKS: If the user's request depends on something that may happen later (for example "if she replies within 4 minutes",
"in exactly N minutes", "each time a new listing appears"), do NOT end your turn after finishing the first part.
Instead: (1) call SystemApp__get_current_time to note the time, (2) call SystemApp__wait_for_notification with a short timeout
(30 seconds or less, so you can react quickly), (3) when it returns, proactively check the relevant app for new messages or changes
(environment changes are NOT pushed to you), (4) act on what you find, and repeat until the time window has passed.
Only send your final report to the user after every time-dependent part of the task has been handled."""

def patch_prompt():
    if not BACKUP.exists(): shutil.copy(PROMPT_FILE, BACKUP)
    src = PROMPT_FILE.read_text()
    anchor = "DEFAULT_ARE_SIMULATION_REACT_JSON_SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format("
    i = src.index(anchor); j = src.index('json_agent_hints="",', i)
    src = src[:j] + "json_agent_hints=" + repr(WAITING_HINT) + "," + src[j + len('json_agent_hints="",'):]
    PROMPT_FILE.write_text(src); print("Waiting instruction added to the system prompt.")

def restore_prompt():
    if BACKUP.exists():
        shutil.copy(BACKUP, PROMPT_FILE); print("System prompt restored.")

patch_prompt()
try:
    run_gaia2(config="time", model="openrouter/google/gemini-2.5-flash", limit=10, run_name="time_flash_wait")
finally:
    restore_prompt()
summarize(["time_flash", "time_flash_wait"])


In [ ]:
# With the instruction: did the agent wait on wezjyh and catch Yujie's replies?
plot_timeline(trace_path("time_flash_wait", "scenario_universe_21_wezjyh"), xmax=260,
              title='Same task, with one added instruction about waiting | Gemini 2.5 Flash')


## Step 18: Download everything (figures and re-judge results included)

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/gaia2_results_v3", "zip", "/content/results")
print("Packed:", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Not in Colab; download the file from the path above.")


# Part 3: Live demo used in the video (D1–D6)
Before recording (not shown): run Steps 1, 2, 2.5, 2.6, 4, 6, 14, the first cell of Step 15, then D1. Clear the outputs of D2–D5.
D3 and D4 run the real agent on one scenario (about a minute each, a few cents).

In [ ]:
# D1: demo helpers
import json, glob, os, shutil, subprocess, shlex, time, textwrap, importlib
from pathlib import Path
from datasets import load_dataset
import are.simulation.agents.default_agent.prompts.system_prompt as sp

FLASH = "openrouter/google/gemini-2.5-flash"
DEMO_ROOT = Path("/content/demo_one")
WEZ = "scenario_universe_21_wezjyh"   # ask Yujie which apartment she likes, wait for her reply, then act

def prepare_single(sid, config="time"):
    # Pull this one scenario from Hugging Face into a local dataset folder so a run covers only it
    d = DEMO_ROOT / sid / config / "validation"
    if d.exists() and list(d.glob("*.json")):
        return DEMO_ROOT / sid
    ds = load_dataset("meta-agents-research-environments/gaia2", name=config, split="validation", streaming=True)
    for row in ds:
        if row.get("scenario_id") == sid:
            d.mkdir(parents=True, exist_ok=True)
            data = row["data"]
            (d / f"{sid}.json").write_text(data if isinstance(data, str) else json.dumps(data))
            return DEMO_ROOT / sid
    raise ValueError(f"scenario not found: {sid}")

PROMPT_FILE = Path(sp.__file__); BACKUP = PROMPT_FILE.with_suffix(".py.bak")
WAITING_HINT = """
TIME-SENSITIVE TASKS: If the user's request depends on something that may happen later (for example "if she replies within 4 minutes",
"in exactly N minutes", "each time a new listing appears"), do NOT end your turn after finishing the first part.
Instead: (1) call SystemApp__get_current_time to note the time, (2) call SystemApp__wait_for_notification with a short timeout
(30 seconds or less, so you can react quickly), (3) when it returns, proactively check the relevant app for new messages or changes
(environment changes are NOT pushed to you), (4) act on what you find, and repeat until the time window has passed.
Only send your final report to the user after every time-dependent part of the task has been handled."""

def patch_prompt():
    if not BACKUP.exists(): shutil.copy(PROMPT_FILE, BACKUP)
    src = BACKUP.read_text()
    anchor = "DEFAULT_ARE_SIMULATION_REACT_JSON_SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format("
    i = src.index(anchor); j = src.index('json_agent_hints="",', i)
    PROMPT_FILE.write_text(src[:j] + "json_agent_hints=" + repr(WAITING_HINT) + "," + src[j + len('json_agent_hints="",'):])

def restore_prompt():
    if BACKUP.exists(): shutil.copy(BACKUP, PROMPT_FILE)

def run_one(sid, tag, config="time", model=FLASH, hint=False, timeout=600):
    root = prepare_single(sid, config)
    out_dir = RESULTS_ROOT / f"demo_{tag}_{time.strftime('%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ["are-benchmark", "run", "--dataset", str(root), "--config", config, "--split", "validation",
           "--model", model, "--provider", "local",
           "--judge_model", JUDGE_MODEL, "--judge_provider", "local",
           "--agent", "default", "--num_runs", "1", "--max_concurrent_scenarios", "1",
           "--scenario_timeout", str(timeout), "--trace_dump_format", "both", "--output_dir", str(out_dir)]
    print("$ " + " ".join(shlex.quote(c) for c in cmd).replace(str(root), "<this one scenario>"))
    print(f"agent: {model.split('/')[-1]}   system prompt: {'default + waiting hint' if hint else 'default'}\n")
    if hint: patch_prompt()
    try:
        t0 = time.time()
        log = open(out_dir / "run.log", "w")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            print(f"\rrunning... {time.time()-t0:5.0f} s", end="")
        p.wait(); log.close()
    finally:
        restore_prompt()
    print(f"\rdone in {time.time()-t0:.0f} s" + " " * 20)
    hits = glob.glob(f"{out_dir}/**/hf/{sid}*.json", recursive=True)
    if not hits:
        print("No trace produced. Last 30 log lines:"); print("".join(open(out_dir / "run.log").readlines()[-30:]))
        return None
    return hits[0]

def _args(a): return {x["name"]: x.get("value") for x in (a or {}).get("args") or [] if isinstance(x, dict)}
WRITES = {"send_message", "send_message_to_group_conversation", "send_email", "remove_saved_apartment",
          "add_calendar_event", "save_apartment", "send_message_to_user"}

def show_task(trace):
    d = json.load(open(trace))
    user = [e for e in d["events"] if e["event_type"] == "USER"][0]
    print("USER REQUEST\n" + textwrap.fill(_args(user["action"]).get("content", ""), 110) + "\n")
    print("WHAT THE WORLD WILL DO (scripted, the agent is not told):")
    for e in d["events"]:
        if e["event_type"] == "ENV":
            a = _args(e["action"])
            print(f"  +{e.get('event_relative_time') or 0:>4.0f} s after the agent messages Yujie:  Yujie replies \"{str(a.get('content',''))[:90]}\"")
    print("\nEXPECTED WRITE ACTIONS (oracle):")
    for e in d["events"]:
        if e["class_name"] == "OracleEvent":
            print("  -", e["action"]["function"])

def show_steps(trace):
    d = json.load(open(trace))
    done = sorted([e for e in d["completed_events"] if e.get("event_time") is not None], key=lambda e: e["event_time"])
    t0 = done[0]["event_time"]
    print("WHAT ACTUALLY HAPPENED")
    for e in done:
        a = e.get("action") or {}; fn = a.get("function") or ""; t = e["event_time"] - t0
        if e["event_type"] == "AGENT":
            mark = "  WRITE" if fn in WRITES else ""
            extra = _args(a).get("content") or _args(a).get("timeout") or ""
            print(f"  t+{t:5.0f}s  agent  {fn:<38}{mark}  {str(extra)[:60]}")
        elif e["event_type"] == "ENV":
            print(f"  t+{t:5.0f}s  WORLD  Yujie replies: \"{str(_args(a).get('content',''))[:70]}\"")
    happened = {e["event_id"] for e in done}
    for e in d["events"]:
        if e["event_type"] == "ENV" and e["event_id"] not in happened:
            print(f"  (never reached) WORLD  Yujie would have replied at +{e.get('event_relative_time') or 0:.0f}s")
    ann = d["metadata"].get("annotation") or {}
    print(f"\nVERDICT: {ann.get('validation_decision')}")

def show_email(run="exec_strong", key="2tof7f"):
    trace = glob.glob(f"/content/results/{run}/hf/*{key}*.json")[0]
    d = json.load(open(trace))
    agent = [e["action"] for e in d["completed_events"] if e["event_type"] == "AGENT" and e["action"].get("function") == "send_email"]
    oracle = [e["action"] for e in d["events"] if e["class_name"] == "OracleEvent" and e["action"].get("function") == "send_email"]
    for name, acts in (("AGENT (Claude Sonnet 4.5) SENT", agent), ("ORACLE EXPECTED", oracle)):
        print(f"===== {name} =====")
        for a in acts:
            x = _args(a); print("to:", x.get("recipients"), "| subject:", x.get("subject")); print(textwrap.fill(str(x.get("content","")), 110), "\n")
    lite = glob.glob(f"/content/results/{run}/lite/*{key}*.json")
    if lite:
        l = json.load(open(lite[0])); print("VERDICT:", l.get("validation_decision"), "|", (l.get("validation_rationale") or "")[:300])

restore_prompt()
_ = prepare_single(WEZ)
print("Demo helpers ready.")

## D2: The scenario

In [ ]:
show_task(trace_path("time_flash", WEZ))

## D3: Live run — default agent

In [ ]:
t_base = run_one(WEZ, "base")
show_steps(t_base)
plot_timeline(t_base, xmax=200, title="Live run | default agent | Gemini 2.5 Flash")

## D4: Same scenario with the waiting instruction

In [ ]:
print(WAITING_HINT, "\n")
t_hint = run_one(WEZ, "hint", hint=True)
show_steps(t_hint)
plot_timeline(t_hint, xmax=260, title="Live run | + waiting instruction | Gemini 2.5 Flash")
print("\nAll 10 Time scenarios (run in Step 17):")
summarize(["time_flash", "time_flash_wait"])

## D5: Can we trust the judge?

In [ ]:
plot_timeline(trace_path("time_strong", "scenario_universe_21_xvc7uo"), xmax=270,
              title='"Notify Linnea whenever an in-budget listing appears" | Claude Sonnet 4.5')
show_email("exec_strong", "2tof7f")
_ = compare_judges("time_flash"); _ = compare_judges("exec_strong")

## D6: Package the demo outputs

In [ ]:
import shutil
zp = shutil.make_archive("/content/gaia2_results_v4", "zip", "/content/results")
from google.colab import files; files.download(zp)